In [1]:
### New

In [2]:
import pandas as pd
from omegaconf import DictConfig
import omegaconf
import psycopg2

In [3]:
cfg = omegaconf.OmegaConf.load("../config.yaml")
conn = psycopg2.connect(**cfg.db)

In [ ]:
# Get searches
def load_searches(snapshot):
    if snapshot == "test":
        snapshot = 1
    query = f"""
    SELECT search_id, serp, queries.query_id, original_id, log_date
    FROM search_logs_normalized
    JOIN queries ON search_logs_normalized.query_id = queries.query_id
    WHERE log_date >= '{cfg["snapshots"][snapshot]["start_date"]}'
        AND log_date <= '{cfg["snapshots"][snapshot]["end_date"]}'
        AND is_structured is FALSE AND is_boolean is FALSE and has_doi is FALSE and was_clicked is TRUE;
    """
    searches = pd.read_sql_query(query, con=conn)
    searches = searches.drop_duplicates(subset=["search_id"])
    searches = searches.rename(columns={"log_date": "search_date"})
    return searches

In [ ]:
def load_clicks(snapshot, searches):
    if snapshot == "test":
        snapshot = 1
    search_ids = list(searches["search_id"].unique())

    query = f"""
    SELECT *
    FROM tracker_logs
    WHERE search_id in {tuple(search_ids)}
        AND log_date >= '{cfg["snapshots"][snapshot]["start_date"]}'
        AND log_date <=  '{cfg["snapshots"][snapshot]["end_date"]}'
        AND (entry_type = 'get-pdf' OR entry_type = 'works')
    """
    clicks = pd.read_sql_query(query, con=conn)
    clicks = clicks.rename(columns={"log_date": "click_date"})

    searches_query_ids = (
        searches[["query_id", "search_id"]].set_index("search_id").to_dict()["query_id"]
    )
    clicks["query_id"] = clicks["search_id"].map(searches_query_ids)
    return clicks

In [ ]:
def filter_candidate_queries(clicks, searches):
    clicks = clicks.merge(
        searches[["search_id", "search_date"]],
        left_on="search_id",
        right_on="search_id",
        how="left",
    )
    print("Clicks before 24h filter:", len(clicks))
    # Only keep clicks that happened within 1 day of the search
    clicks = clicks[
        clicks["click_date"] <= clicks["search_date"] + pd.Timedelta(days=1)
    ]
    print("Clicks after 24h filter:", len(clicks))

    click_stats = (
        clicks.groupby("query_id").count().reset_index()[["query_id", "search_id"]]
    )
    candidate_queries = click_stats[click_stats["search_id"] > 30]
    print("Candidate queries with at least 30 clicks:", len(candidate_queries))
    return candidate_queries

In [ ]:
snapshot = 3
searches = load_searches(snapshot)
clicks = load_clicks(snapshot, searches)
candidate_queries = filter_candidate_queries(clicks, searches)

### Candidate query stats:
**Snapshot 1:**
- Clicks before 24h filter: 552032
- Clicks after 24h filter: 390315
- Candidate queries with at least 30 clicks: 463

**Snapshot 2:**
- Clicks before 24h filter: 572471
- Clicks after 24h filter: 224763
- Candidate queries with at least 30 clicks: 218

**Snapshot 3:**
- Clicks before 24h filter: 3194586
- Clicks after 24h filter: 1821807
- Candidate queries with at least 30 clicks: 240